# Qwen3-4B Fine-tuning v4 — CPT then SFT for Knowledge Injection

## What changed vs v3 and why

v3's symptoms: train_loss drops < 1.0, eval_loss stuck at ~1.5, generic answers at inference. Root causes that v4 fixes:

| Bug in v3 | Symptom it caused | v4 fix |
|---|---|---|
| CPT did ~56 total steps (LR=5e-4, 10 epochs, ~90 char-chunks, batch 16) | Knowledge never absorbed | Token-based chunking + 50 epochs at eff_batch=8 → ~350 CPT steps |
| Char-based chunking cuts mid-token and ignores structure | Garbled chunk boundaries | Token-based sliding window (768 tokens, 192 overlap) |
| LoRA r=32 too small to encode 46K-token report + override Instruct prior | Generic answers at inference | r=64, alpha=128 (high scale), target all linear modules |
| `use_dora=True` + `use_rslora=True` is undocumented; DoRA can silently no-op | Unstable / silent capacity loss | Standard LoRA with rsLoRA only |
| `packing=True` + `train_on_responses_only` causes mask leaks across packed examples | Loss signal corrupted in SFT | `packing=False` for SFT (CPT keeps packing) |
| EarlyStopping with patience=4 + eval_steps=5 fires at ~step 30-60 | SFT halted before it converges | patience=8, eval_steps=10, min_evals=8 |
| `metric_for_best_model="eval_loss"` chases an unreachable target | Stops at wrong checkpoint | Same metric BUT realistic: target eval_loss 0.7–0.9, judge by generation quality |
| No before/after generation check | Couldn't tell if LoRA was being applied | Cell tests 3 questions BEFORE training, AFTER CPT, AFTER SFT |

## Realistic targets for THIS dataset

Your eval set holds out 10% of Q&A pairs. Many held-out questions ask about facts whose exact wording the model never saw → predicting the exact answer tokens is mathematically bounded. **eval_loss < 0.5 is not achievable on this kind of held-out QA set without contamination.** Realistic:

- **CPT train_loss:** 1.5 → 0.4 (memorizing the report text)
- **SFT train_loss:** 0.6 → 0.25 (memorizing answer phrasings)
- **SFT eval_loss:** 0.7 – 0.95 (good = factual content present, just paraphrased)
- **Inference quality:** specific numbers/names from the report, not generic platitudes — **this is the real test.**

In [1]:
# ── Cell 1: Install ──────────────────────────────────────────────────────────
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

UsageError: Line magic function `%%capture` not found.


In [2]:
%pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 MB 24.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 94.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.1/721.1 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 85.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 90.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 6.6 MB/s eta 0:00:00
   ━

In [3]:
import warnings
warnings.filterwarnings("ignore")

In [4]:
# ── Cell 2: Mount Drive ──────────────────────────────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')

In [5]:
# ── Cell 3: Paths & Config ───────────────────────────────────────────────────
BASE_DIR    = "/kaggle/input/datasets/sasidhar77/dataset-synthetic"
CKT_DIR = "/kaggle/working/"
DATA_PATH   = f"{BASE_DIR}/Global_outlok.json"
RAW_TEXT    = f"{BASE_DIR}/global_outlook_raw_text.txt"

CPT_DIR     = f"{CKT_DIR}/qwen3_v4_cpt_ckpts"
SFT_DIR     = f"{CKT_DIR}/qwen3_v4_sft_ckpts"
MERGED_DIR  = f"{CKT_DIR}/qwen3_v4_merged"

print(f"Raw text: {RAW_TEXT}")
print(f"Q&A:      {DATA_PATH}")
print(f"CPT out:  {CPT_DIR}")
print(f"SFT out:  {SFT_DIR}")
print(f"Merged:   {MERGED_DIR}")

Raw text: /kaggle/input/datasets/sasidhar77/dataset-synthetic/global_outlook_raw_text.txt
Q&A:      /kaggle/input/datasets/sasidhar77/dataset-synthetic/Global_outlok.json
CPT out:  /kaggle/working//qwen3_v4_cpt_ckpts
SFT out:  /kaggle/working//qwen3_v4_sft_ckpts
Merged:   /kaggle/working//qwen3_v4_merged


In [6]:
# ── Cell 4: Load Model ───────────────────────────────────────────────────────
# Using Instruct variant (Qwen3-4B-Instruct-2507). For CPT-then-SFT, the Base
# variant is sometimes cleaner — but Instruct works fine if the LoRA has
# enough capacity (r=64 below) to override the 'be generic' prior.
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = "unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length  = 2048,
    load_in_4bit    = True,
    load_in_8bit    = False,
    full_finetuning = False,
)

gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name} | VRAM: {gpu.total_memory/1024**3:.1f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.48k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/4.04k [00:00<?, ?B/s]

unsloth/qwen3-4b-instruct-2507-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
GPU: Tesla T4 | VRAM: 14.6 GB


In [7]:
# ── Cell 5: LoRA Adapter (used for BOTH CPT and SFT — same adapter) ──────────
#
# Why r=64 alpha=128:
#   - r=32 in v3 wasn't enough capacity to (a) memorize a 46K-token report AND
#     (b) override the Instruct model's strong 'give a generic answer' prior.
#   - r=64 doubles capacity. With rsLoRA, effective scale = alpha/sqrt(r) = 128/8 = 16,
#     which is meaningful but stable.
#   - Targeting all 7 linear modules (attn + MLP) is correct; embed_tokens/lm_head
#     are NOT included on purpose — adding them on a 4B Instruct model with so
#     little data destabilizes generation.
#
# Why NOT use_dora here:
#   DoRA + rsLoRA + Unsloth has known compatibility issues. Standard LoRA with
#   rsLoRA is well-tested and gives essentially the same generalization for
#   knowledge-injection tasks.
#
# Why dropout=0.05:
#   For knowledge memorization (CPT), we WANT memorization; high dropout
#   actively hurts. Keep it low.

model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha     = 32,
    lora_dropout   = 0.05,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 3407,
    use_rslora     = True,
    loftq_config   = None,
)
model.print_trainable_parameters()

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.6.9 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


In [8]:
# ── Cell 6: Chat Template ────────────────────────────────────────────────────
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template="qwen3-instruct")

In [9]:
# ── Cell 7: Inference helper (used at every checkpoint to verify learning) ───
SYSTEM_PROMPT = (
    "You are a specialist analyst for the Dun & Bradstreet Global Outlook 2026 report. "
    "Your answers must come directly from that report's content, data, and projections. "
    "Be specific and precise — use exact figures, country names, and report terminology."
)

PROBE_QUESTIONS = [
    "What GDP growth rate did Dun & Bradstreet forecast for Sweden in 2026?",
    "How did Sweden's economy perform in 2025 compared to what was expected in 2026?",
    "What does the Global Outlook 2026 identify as the main swing factor for India's economic performance?",
]

def ask(question: str, max_new_tokens: int = 300) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None, top_p=None, top_k=None,
            repetition_penalty=1.1,
        )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def probe(label: str):
    FastLanguageModel.for_inference(model)
    print(f"\n{'='*78}\n=== PROBE: {label}\n{'='*78}")
    for q in PROBE_QUESTIONS:
        print(f"\nQ: {q}")
        print(f"A: {ask(q, 200)}")
    print("="*78)

# Baseline check — model should give generic answers here, NOT report specifics.
probe("BASELINE (no training yet)")


=== PROBE: BASELINE (no training yet)

Q: What GDP growth rate did Dun & Bradstreet forecast for Sweden in 2026?


Both `max_new_tokens` (=200) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: Dun & Bradstreet did not provide a specific GDP growth rate forecast for Sweden in their Global Outlook 2026 report.

The available information in the report focuses on broader global trends, regional economic outlooks (such as Europe), and macroeconomic indicators like inflation, trade, and investment flows. While Sweden is part of the European region and may be referenced in context with EU or Nordic performance, no standalone GDP growth projection specifically for Sweden in 2026 was included in the report.

Therefore, based on the provided content: **No GDP growth rate for Sweden in 2026 was forecasted by Dun & Bradstreet in the Global Outlook 2026 report**.

Q: How did Sweden's economy perform in 2025 compared to what was expected in 2026?


Both `max_new_tokens` (=200) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: The Dun & Bradstreet Global Outlook 2026 report does not provide specific performance data on Sweden’s economy for 2025 or compare it directly to expectations for 2026. As of now, there is no publicly available information within the report indicating how Sweden’s economy performed in 2025 relative to its projected performance in 2026.

Therefore, based on the available content and structure of the Dun & Bradstreet Global Outlook 2026 report, **no comparative analysis between Sweden’s actual 2025 performance and its 2026 expectations is provided**.

If you have access to a specific section or data table from the report related to Sweden or Nordic economies, I can help interpret that content. Otherwise, this question cannot be answered with confidence using only the known scope of the report.

Q: What does the Global Outlook 2026 identify as the main swing factor for India's economic performance?
A: According to the *Dun & Bradstreet Global Outlook 2026*, the main swing factor for In

---
## Phase 1: Continued Pre-Training (CPT)

**Goal:** Make the model memorize the report's text, numbers, and concepts.

**Why this matters:** SFT alone teaches the model to answer Q&A pairs but never shows it the source text. With only 1,430 (Q,A) pairs, each fact is seen in one specific phrasing — the model memorizes that phrasing without internalizing the underlying knowledge. CPT first reads the whole report many times so the facts are encoded in weights; SFT then teaches retrieval-style behavior.

**What the math says we need:** Raw text is ~35K tokens → ~45 windows of 768 tokens. To inject knowledge: ≥300 optimizer steps. With eff_batch=8 → 50 epochs gets us ~280 steps — sufficient for memorization.

In [10]:
# ── Cell 8: Prepare CPT dataset (token-based sliding window) ─────────────────
from datasets import Dataset

with open(RAW_TEXT) as f:
    raw_text = f.read()

# Token-based chunking. The v3 char-based version cut mid-token and threw away
# structure — this version produces clean, full-token windows.
all_token_ids = tokenizer.encode(raw_text, add_special_tokens=False)
print(f"Raw text token count: {len(all_token_ids):,}")

WINDOW = 768   # tokens per training example
STRIDE = 576   # 25% overlap (192 tokens) — preserves cross-paragraph context

chunks = []
for start in range(0, len(all_token_ids), STRIDE):
    window_ids = all_token_ids[start:start + WINDOW]
    if len(window_ids) < 128:        # skip tail too small to be useful
        break
    chunks.append({"text": tokenizer.decode(window_ids, skip_special_tokens=False)})
    if start + WINDOW >= len(all_token_ids):
        break

cpt_dataset = Dataset.from_list(chunks)
print(f"CPT chunks: {len(cpt_dataset)} (window={WINDOW}, stride={STRIDE})")
print(f"\n--- First chunk preview ---\n{cpt_dataset[0]['text'][:400]}...")

Raw text token count: 37,769
CPT chunks: 66 (window=768, stride=576)

--- First chunk preview ---
Adaptation Over Acceleration: Adaptation Over Acceleration: The Global Economy in 2026 The Global Economy in 2026

## Contents 

03 Executive Summary 

04 Key Takeaways 

10 The World in 2026 

42 Global Sectoral Outlook 

60 Thought Leadership II: Global AI 

65 Thought Leadership III: Global Supply Chains in 2026 


The content is a simple graphic with a blue triangular shape on a white backgrou...


In [11]:
# ── Cell 9: Run CPT ──────────────────────────────────────────────────────────
#
# Hyperparameter rationale (changes from v3 in [brackets]):
#
#   learning_rate = 2e-4         [v3: 5e-4]
#     5e-4 was too aggressive for stable memorization on tiny corpus.
#     2e-4 is the standard CPT-on-LoRA value.
#
#   num_train_epochs = 50        [v3: 10]
#     With ~45 chunks at eff_batch=8: 50 epochs ≈ 280 steps. v3's 10 epochs
#     gave only ~56 steps — far below the threshold for knowledge to stick.
#
#   per_device=2, grad_accum=4 → eff_batch=8     [v3: eff_batch=16]
#     Smaller eff_batch = more optimizer steps per epoch, which we need on
#     this small corpus.
#
#   warmup_ratio = 0.05          [v3: 0.10]
#     Total run is short; a long warmup wastes steps.
#
#   weight_decay = 0.0           [v3: 0.01]
#     We WANT memorization here. WD pulls weights toward zero — the opposite
#     of what knowledge injection needs. Save WD for SFT.
#
#   packing = True               [unchanged]
#     Correct for CPT (no response-only masking, full causal LM loss).
#
from trl import SFTTrainer, SFTConfig

cpt_trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = cpt_dataset,
    args = SFTConfig(
        dataset_text_field          = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs            = 50,
        learning_rate               = 2e-4,
        lr_scheduler_type           = "cosine",
        warmup_ratio                = 0.05,
        weight_decay                = 0.0,
        logging_steps               = 10,
        save_strategy               = "epoch",
        save_total_limit            = 2,
        optim                       = "adamw_8bit",
        packing                     = True,
        max_seq_length              = 1024,
        seed                        = 3407,
        report_to                   = "none",
        output_dir                  = CPT_DIR,
    ),
)

print("Starting CPT...")
cpt_stats = cpt_trainer.train()
print(f"\nCPT done. Final train loss: {cpt_stats.training_loss:.4f}")
print("Target: <0.5  (anything <0.7 means the model has absorbed the text)")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/66 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=8):   0%|          | 0/66 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!
Starting CPT...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 66 | Num Epochs = 50 | Total steps = 450
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,2.466579
20,1.995406
30,1.640965
40,1.167134
50,0.691856
60,0.360596
70,0.160533
80,0.097883
90,0.070471
100,0.056286


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working//qwen3_v4_cpt_ckpts/checkpoint-9/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working//qwen3_v4_cpt_ckpts/checkpoint-18/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working//qwen3_v4_cpt_ckpts/checkpoint-27/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working//qwen3_v4_cpt_ckpts/checkpoint-36/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working//qwen3_v4_cpt_ckpts/checkpoint-45/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working//qwen3_v4_cpt_ckpts/checkpoint-54/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working//qwen3_v4_cpt_ckpts/checkpoint-63/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working//qwen3_v4_cpt_ckpts/checkpoint-72/tokenizer_config.json.
Unsloth: 


CPT done. Final train loss: 0.1992
Target: <0.5  (anything <0.7 means the model has absorbed the text)


In [12]:
# ── Cell 10: Probe after CPT ─────────────────────────────────────────────────
# After CPT the model has READ the report but hasn't been taught to ANSWER
# questions about it. Outputs may be raw report-style continuations rather
# than direct answers — that's expected and fine. What you should see:
#   - Specific numbers/country names from the report appearing in outputs
#   - Vocabulary shifting toward report terminology
#   - NOT yet clean Q&A formatting
probe("AFTER CPT (knowledge injected, not yet Q&A-formatted)")

Both `max_new_tokens` (=200) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



=== PROBE: AFTER CPT (knowledge injected, not yet Q&A-formatted)

Q: What GDP growth rate did Dun & Bradstreet forecast for Sweden in 2026?


Both `max_new_tokens` (=200) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: According to the Dun & Bradstreet Global Outlook 2026, Dun & Bradstreet forecasts that Sweden will have a real GDP growth rate of **1.9%** in 2026. 


Source: Dun & Bradstreet Global Outlook 2026 – Page 19 (Sweden section)

Q: How did Sweden's economy perform in 2025 compared to what was expected in 2026?


Both `max_new_tokens` (=200) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: According to the Dun & Bradstreet Global Outlook 2026 report, Sweden’s economy performed below expectations in 2025 and is projected to continue its slow recovery in 2026. 

In 2025, real GDP growth in Sweden was close to 0%, which was below the expected 1.5% growth. This underperformance followed an above-average 2024 outlook, highlighting a slowdown in momentum. The 2026 projection for Sweden is also cautious, with growth expected at 0.8%, but this figure is lower than the overall EU average, indicating lingering constraints. 

The main factors limiting Sweden’s recovery in 2025–2026 include rigid labor market conditions, restricted household spending due to high inequality and tax burdens, and weak business investment plans. These issues have kept headline inflation elevated (at 2.1%) and have constrained policy support, further slowing economic expansion. 

In

Q: What does the Global Outlook 2026 identify as the main swing factor for India's economic performance?
A: According t

---
## Phase 2: Supervised Fine-Tuning (SFT)

**Goal:** Teach the (now-knowledgeable) model to answer questions in the right format.

Same LoRA adapter as CPT — we continue training it. SFT now needs to do much less work because the knowledge is already in the weights.

In [12]:
# ── Cell 11: Prepare SFT dataset (prompt/completion pairs) ───────────────────
import json, random
from sklearn.model_selection import train_test_split
from datasets import Dataset
random.seed(42)

with open(DATA_PATH) as f:
    raw = json.load(f)

# raw is a list of {"prompt": ..., "completion": ...} dicts
indices = list(range(len(raw)))
train_idx, eval_idx = train_test_split(indices, test_size=0.10, random_state=42)

def fmt_prompt_completion(examples):
    texts = []
    for prompt, completion in zip(examples["prompt"], examples["completion"]):
        # Format as instruct-style: user turn = prompt, assistant turn = completion
        messages = [
            {"role": "user",      "content": prompt},
            {"role": "assistant", "content": completion},
        ]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False, enable_thinking=False
        )
        texts.append(text)
    return {"text": texts}

train_records = [raw[i] for i in train_idx]
eval_records  = [raw[i] for i in eval_idx]

train_ds = Dataset.from_list(train_records).map(fmt_prompt_completion, batched=True)
eval_ds  = Dataset.from_list(eval_records).map(fmt_prompt_completion, batched=True)
print(f"SFT train: {len(train_ds)} | eval: {len(eval_ds)}")
print("\nSample formatted text:")
print(train_ds[0]["text"][:400])


Map:   0%|          | 0/8383 [00:00<?, ? examples/s]

Map:   0%|          | 0/715 [00:00<?, ? examples/s]

SFT train: 8383 (1948 augmented) | eval: 715


In [13]:
# ── Cell 12: Custom early-stopping callback (gap-aware) ──────────────────────
from transformers import TrainerCallback

class GapMonitorEarlyStopping(TrainerCallback):
    """Stops if eval doesn't improve for `patience` evals OR if eval-train gap > threshold."""
    def __init__(self, patience=8, gap_threshold=0.6, min_evals=8):
        self.patience, self.gap_threshold, self.min_evals = patience, gap_threshold, min_evals
        self.best_eval = float("inf")
        self.no_improve = 0
        self.eval_count = 0
        self.last_train_loss = None

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs and "eval_loss" not in logs:
            self.last_train_loss = logs["loss"]

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if not metrics or "eval_loss" not in metrics:
            return
        eval_loss = metrics["eval_loss"]
        self.eval_count += 1

        if eval_loss < self.best_eval - 1e-4:
            self.best_eval, self.no_improve = eval_loss, 0
        else:
            self.no_improve += 1

        gap = (eval_loss - self.last_train_loss) if self.last_train_loss is not None else None
        gap_str = f"{gap:.3f}" if gap is not None else "N/A"
        print(f"[ES] eval#{self.eval_count} eval={eval_loss:.4f} "
              f"train={self.last_train_loss or 'N/A'} gap={gap_str} "
              f"no_improve={self.no_improve}/{self.patience}")

        if self.eval_count < self.min_evals:
            return
        if self.no_improve >= self.patience:
            print(f"  ⛔ STOP: no improvement for {self.patience} evals (best={self.best_eval:.4f})")
            control.should_training_stop = True
        elif gap is not None and gap > self.gap_threshold:
            print(f"  ⛔ STOP: gap {gap:.3f} > {self.gap_threshold}")
            control.should_training_stop = True

In [14]:
# ── Cell 13: SFT trainer ─────────────────────────────────────────────────────
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

EFF_BATCH = 16  # per_device=2 × grad_accum=8
steps_per_epoch = max(1, len(train_ds) // EFF_BATCH)
print(f"Steps/epoch: ~{steps_per_epoch}, eval every 25 → ~{steps_per_epoch // 25} evals/epoch")

sft_trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_ds,
    eval_dataset  = eval_ds,
    args = SFTConfig(
        dataset_text_field          = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,
        num_train_epochs            = 4,
        learning_rate               = 5e-5,
        lr_scheduler_type           = "cosine",
        warmup_ratio                = 0.10,
        weight_decay                = 0.05,
        logging_steps               = 5,
        eval_strategy               = "steps",
        eval_steps                  = 25,
        save_strategy               = "steps",
        save_steps                  = 25,
        save_total_limit            = 3,
        load_best_model_at_end      = True,
        metric_for_best_model       = "eval_loss",
        greater_is_better           = False,
        optim                       = "adamw_8bit",
        packing                     = False,
        max_seq_length              = 2048,
        seed                        = 3407,
        report_to                   = "none",
        output_dir                  = SFT_DIR,
    ),
)
# Mask prompt tokens — only train on completion (assistant) tokens
sft_trainer = train_on_responses_only(
    sft_trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)
sft_trainer.add_callback(GapMonitorEarlyStopping(patience=8, gap_threshold=0.6, min_evals=8))

# Verify masking
sample_labels = sft_trainer.train_dataset[0]["labels"]
decoded = tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in sample_labels])
print("\nMasking check (PAD = ignored, real tokens = completion only):")
print(decoded[:600])


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Steps/epoch: ~523, eval every 25 → ~20 evals/epoch


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/8383 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/715 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/8383 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/8383 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/715 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/715 [00:00<?, ? examples/s]


Masking check (PAD = ignored, real tokens = assistant response):
<|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|PAD_TOKEN|><|


In [15]:
# ── Cell 14: Run SFT ─────────────────────────────────────────────────────────
sft_stats = sft_trainer.train()
print(f"\nBest checkpoint: {sft_trainer.state.best_model_checkpoint}")
print(f"Best eval_loss:  {sft_trainer.state.best_metric:.4f}")
print("\nInterpretation:")
print("  eval_loss < 0.7  → great (model knows the content well)")
print("  eval_loss 0.7-0.95 → good (model knows facts; eval set has unseen phrasings)")
print("  eval_loss > 1.1  → CPT didn't take; rerun CPT with more epochs")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8,383 | Num Epochs = 4 | Total steps = 2,096
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
25,3.201380,2.789136
50,2.118407,2.028183
75,2.012334,1.841951
100,1.827973,1.739704
125,1.789885,1.670480
150,1.633972,1.617264
175,1.527366,1.558382
200,1.554213,1.506630
225,1.459259,1.459704
250,1.399372,1.411579


[ES] eval#1 eval=2.7891 train=3.201380157470703 gap=-0.412 no_improve=0/8


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working//qwen3_v4_sft_ckpts/checkpoint-25/tokenizer_config.json.


...

[ES] eval#47 eval=0.8504 train=0.3803746223449707 gap=0.470 no_improve=7/8


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working//qwen3_v4_sft_ckpts/checkpoint-1175/tokenizer_config.json.


[ES] eval#48 eval=0.8453 train=0.3801807641983032 gap=0.465 no_improve=8/8
  ⛔ STOP: no improvement for 8 evals (best=0.8394)


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working//qwen3_v4_sft_ckpts/checkpoint-1200/tokenizer_config.json.



Best checkpoint: /kaggle/working//qwen3_v4_sft_ckpts/checkpoint-1000
Best eval_loss:  0.8394

Interpretation:
  eval_loss < 0.7  → great (model knows the content well)
  eval_loss 0.7-0.95 → good (model knows facts; eval set has unseen phrasings)
  eval_loss > 1.1  → CPT didn't take; rerun CPT with more epochs


In [ ]:
# ── Cell 15: Probe after SFT ─────────────────────────────────────────────────
probe("AFTER SFT (final model)")

In [ ]:
def ask1(question: str, max_new_tokens: int = 300) -> str:
    messages = [
        # {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None, top_p=None, top_k=None,
            repetition_penalty=1.1,
        )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


In [ ]:
ask("How did Sweden's economy perform in 2025 compared to what was expected in 2026?")

In [ ]:
# ── Cell 16: Plot loss curves ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

history = sft_trainer.state.log_history
tr_steps, tr_loss, ev_steps, ev_loss = [], [], [], []
for e in history:
    if 'loss' in e and 'eval_loss' not in e:
        tr_steps.append(e['step']); tr_loss.append(e['loss'])
    if 'eval_loss' in e:
        ev_steps.append(e['step']); ev_loss.append(e['eval_loss'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(tr_steps, tr_loss, label='train', color='steelblue', alpha=0.7)
axes[0].plot(ev_steps, ev_loss, label='eval',  color='tomato', linewidth=2, marker='o')
axes[0].axhline(y=0.7, color='green',  linestyle='--', alpha=0.5, label='target eval')
axes[0].set_xlabel('Steps'); axes[0].set_ylabel('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_title('SFT loss curves')

if len(ev_steps) > 1:
    tr_interp = np.interp(ev_steps, tr_steps, tr_loss)
    gap = [e - t for e, t in zip(ev_loss, tr_interp)]
    axes[1].plot(ev_steps, gap, color='purple', linewidth=2, marker='o')
    axes[1].axhline(y=0.6, color='red',    linestyle='--', label='stop threshold')
    axes[1].axhline(y=0.4, color='orange', linestyle='--', label='warning')
    axes[1].set_xlabel('Steps'); axes[1].set_ylabel('eval - train'); axes[1].legend(); axes[1].grid(alpha=0.3)
    axes[1].set_title('Generalization gap')

plt.tight_layout()
plt.savefig(f"{SFT_DIR}/curves.png", dpi=120)
plt.show()

In [ ]:
# ── Cell 17: Quantitative evaluation on held-out questions ───────────────────
import re
from tqdm import tqdm

FastLanguageModel.for_inference(model)

def token_recall(pred, ref):
    p = set(pred.lower().split()); r = ref.lower().split()
    return sum(1 for t in r if t in p) / max(1, len(r))

def number_recall(pred, ref):
    nums = re.findall(r'\d+\.?\d*', ref)
    return 1.0 if not nums else sum(1 for n in nums if n in pred) / len(nums)

sample = eval_convs[:50]
tr, nr = [], []
for c in tqdm(sample, desc="eval"):
    pred = ask(c[1]['content'], 350)
    tr.append(token_recall(pred, c[2]['content']))
    nr.append(number_recall(pred, c[2]['content']))

print(f"\nHeld-out evaluation ({len(sample)} questions):")
print(f"  Token recall  : {sum(tr)/len(tr):.3f}  (>0.45 = good, >0.60 = excellent)")
print(f"  Number recall : {sum(nr)/len(nr):.3f}  (>0.65 = good, >0.80 = excellent)")
print("\nLowest 5 (where model fails):")
for s, c in sorted(zip(tr, sample), key=lambda x: x[0])[:5]:
    print(f"  recall={s:.2f} | {c[1]['content'][:90]}")

In [ ]:
# ── Cell 18: Save merged model ───────────────────────────────────────────────
model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")
print(f"Saved → {MERGED_DIR}")

---
## If results are still weak after this run

Check in this order:

1. **Did CPT train_loss reach < 0.7?** If not, CPT didn't absorb the text. Solutions:
   - Increase CPT epochs to 80
   - Drop CPT eff_batch to 4 (per_device=1, grad_accum=4) → 2× more steps
   - Confirm `RAW_TEXT` actually contains the report content (not just the cover/TOC)

2. **Does the AFTER-CPT probe show report-specific vocabulary?** If output is still generic post-CPT, the model is ignoring the adapter. Solutions:
   - Bump rank to r=128 (uses ~2GB more VRAM — verify on T4)
   - Increase `lora_alpha` to 256 (effective scale doubles)

3. **Does SFT eval_loss drop below 1.0?** If not, the held-out questions test facts that even CPT didn't absorb (likely tail content). Solutions:
   - Run CPT for another 30 epochs starting from the saved CPT checkpoint
   - Reduce `test_size` from 0.10 to 0.05 (more training signal)

4. **Are inference outputs accurate but truncated?** Bump `max_new_tokens` to 500.

5. **Is the fundamental task too hard for 4B?** A 46K-token report contains thousands of facts. 4B has limited memorization capacity. Two escapes:
   - Move to Qwen3-7B or 14B (won't fit on T4 — needs ≥A100)
   - **RAG instead of fine-tuning**: embed the report with a retriever, retrieve top-k chunks at inference, let the model summarize. Much more reliable for factual QA on a fixed corpus.